# 1D J1J2J3 inference: PoincareGRU (seed 111-555)

This is part of the work arxiv:2604.24337 [quant-ph] by HL Dao. For the purpose of reproducing the results, the paths to the trained weights have to be adjusted to match the repo structure.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.append('../../1_hypnqs_torch_poincare/utility_poincare')
from j1j2j3_hyprnn_train_loop import *
numsamples = 10000

GPU Available:  False


In [2]:
def set_cpu_deterministic(seed):
    # 1. Python & Numpy
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 2. PyTorch CPU
    torch.manual_seed(seed)
    
    # 3. Force Deterministic Algorithms
    # This prevents non-deterministic CPU operations (like some views/reductions)
    torch.use_deterministic_algorithms(True)
    
    # 4. Limit CPU Threads
    # Setting this to 1 ensures operations are done in a fixed order.
    torch.set_num_threads(1)

In [3]:
def clip_local_energies(eloc, threshold=5.0):
    # Convert to numpy if it's a torch tensor, or vice versa
    eloc_real = np.real(eloc)
    median = np.median(eloc_real)
    mad = np.median(np.abs(eloc_real - median))
    
    # Standard safety check to avoid division by zero if MAD is 0
    if mad == 0:
        return eloc
        
    lower_bound = median - threshold * mad
    upper_bound = median + threshold * mad
    
    # Clip the values (keeping the imaginary part if it exists)
    # We create a copy to avoid modifying the original array in place
    clipped = np.clip(eloc_real, lower_bound, upper_bound)
    
    # If the original was complex, restore the imaginary part
    if np.iscomplexobj(eloc):
        return clipped + 1j * np.imag(eloc)
    return clipped

def define_load_test(wf,  numsamples,path_to_weights, Ee, clipped_e = False):
    state_dict = torch.load(path_to_weights, map_location=torch.device('cpu'))   
    new_state_dict = {}
    for key, value in state_dict.items():
        # Strip prefixes and rename keys to match current architecture
        new_key = key.replace('model.', '').replace('cell.', 'rnn.')
        new_state_dict[new_key] = value
    # This line loads the RE-MAPPED weights
    wf.model.load_state_dict(new_state_dict, strict=False)
    print("Successfully remapped and loaded weights.")
    
    # --- PART C: Check performance AFTER loading ---
    with torch.no_grad():
        test_samples_after = wf.sample(numsamples)
        if clipped_e:
            # 1. Get raw energies
            raw_gs_after = J1J2J3_local_energies(wf, N, J1, J2,J3, Bz, numsamples, test_samples_after, Marshall_sign=True)
    
            # 2. APPLY CLIPPING
            test_gs_after = clip_local_energies(raw_gs_after, threshold=5.0)
    
            # 3. Calculate statistics on cleaned data
            gs_mean_a = np.round(np.mean(test_gs_after), 4)
            gs_var_a = np.round(np.var(test_gs_after), 4)
    
            # Optional: Count how many were clipped to see if the model is unstable
            num_clipped = np.sum(np.real(raw_gs_after) != np.real(test_gs_after))
            print(f"Clipped {num_clipped} outlier samples out of {numsamples}")
        else:
            test_gs_after =  J1J2J3_local_energies(wf, N, J1, J2, J3, Bz, numsamples, test_samples_after, Marshall_sign=True)
            gs_mean_a = np.round(np.mean(test_gs_after),4)
            gs_var_a = np.round(np.var(test_gs_after),4)
    
    #wf.model.summary()
    #print('====================================================================')
    print(f'After loading weights, the ground state energy mean and variance are:')
    print(f'Mean E = {gs_mean_a}, var E = {gs_var_a}')
    print(f'DMRG energy  is {np.round(Ee,4)}')

In [4]:
N=100
syssize = 100
J1_ = 1.0
J1 = +J1_ * np.ones(syssize)
Bz = +0.0 * np.ones(syssize)
fname=f'result_PoincareGRU'

## J2=0.0, J3=0.5, rmax=1.0

In [4]:
J2_ = 0.0
J3_ = 0.5
r_max=1.0
J2 = +J2_ * np.ones(syssize)
J3 = +J3_ * np.ones(syssize)
Ee_00_05= -53.99140745384424

In [5]:
units = 70
seed=111
hGRU = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max=r_max, seed=seed)
h_wl2 = f'J2={J2_}_J3={J3_}_rmax=1.0/seed={seed}/N100_J1=1.0|J2=0.0|J3=0.5_HypGRU_70_id_hyp_rmax={r_max}_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU, numsamples, h_wl2, Ee_00_05, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 519 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-52.82870101928711-0.032999999821186066j), var E = 14.668700218200684
DMRG energy  is -53.9914
Time taken=0.417 hrs


In [6]:
units = 70
seed=222
hGRU = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max=r_max, seed=seed)
h_wl2 = f'J2={J2_}_J3={J3_}_rmax=1.0/seed={seed}/N100_J1=1.0|J2=0.0|J3=0.5_HypGRU_70_id_hyp_rmax={r_max}_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU, numsamples, h_wl2, Ee_00_05, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 627 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-52.66830062866211+0.005900000222027302j), var E = 2.5355000495910645
DMRG energy  is -53.9914
Time taken=0.404 hrs


In [7]:
units = 70
seed=333
hGRU = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max=r_max, seed=seed)
h_wl2 = f'J2={J2_}_J3={J3_}_rmax=1.0/seed={seed}/N100_J1=1.0|J2=0.0|J3=0.5_HypGRU_70_id_hyp_rmax={r_max}_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU, numsamples, h_wl2, Ee_00_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-53.5432014465332-0.01730000041425228j), var E = 4.037899971008301
DMRG energy  is -53.9914
Time taken=0.414 hrs


In [8]:
units = 70
seed=444
hGRU = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max=r_max, seed=seed)
h_wl2 = f'J2={J2_}_J3={J3_}_rmax=1.0/seed={seed}/N100_J1=1.0|J2=0.0|J3=0.5_HypGRU_70_id_hyp_rmax={r_max}_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU, numsamples, h_wl2, Ee_00_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-53.4281005859375-0.009499999694526196j), var E = 1.4127999544143677
DMRG energy  is -53.9914
Time taken=0.425 hrs


In [9]:
units = 70
seed=555
hGRU = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max=r_max, seed=seed)
h_wl2 = f'J2={J2_}_J3={J3_}_rmax=1.0/seed={seed}/N100_J1=1.0|J2=0.0|J3=0.5_HypGRU_70_id_hyp_rmax={r_max}_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU, numsamples, h_wl2, Ee_00_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-53.44559860229492+0.000699999975040555j), var E = 1.7955000400543213
DMRG energy  is -53.9914
Time taken=0.461 hrs


## J2=0.2, J3=0.2, rmax=1.0

In [10]:
J2_ = 0.2
J3_ = 0.2
J2 = +J2_ * np.ones(syssize)
J3 = +J3_ * np.ones(syssize)
Ee_02_02=-43.58595579948936
r_max=1.0

In [11]:
units = 70
seed=111
hGRU = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max = r_max, seed=seed)
h_wl2 = f'J2={J2_}_J3={J3_}_rmax={r_max}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.2_HypGRU_70_id_hyp_rmax=1.0_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU, numsamples, h_wl2, Ee_02_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 574 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-42.88949966430664+9.999999747378752e-05j), var E = 0.6872000098228455
DMRG energy  is -43.586
Time taken=0.465 hrs


In [12]:
units = 70
seed=222
hGRU = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max = r_max, seed=seed)
h_wl2 = f'J2={J2_}_J3={J3_}_rmax={r_max}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.2_HypGRU_70_id_hyp_rmax=1.0_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU, numsamples, h_wl2, Ee_02_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 358 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-42.3494987487793-0.0026000000070780516j), var E = 0.6248999834060669
DMRG energy  is -43.586
Time taken=0.461 hrs


In [13]:
units = 70
seed=333
hGRU = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max = r_max, seed=seed)
h_wl2 = f'J2={J2_}_J3={J3_}_rmax={r_max}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.2_HypGRU_70_id_hyp_rmax=1.0_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU, numsamples, h_wl2, Ee_02_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 472 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-42.41830062866211+0.026900000870227814j), var E = 1.4354000091552734
DMRG energy  is -43.586
Time taken=0.448 hrs


In [14]:
units = 70
seed=444
hGRU = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max = r_max, seed=seed)
h_wl2 = f'J2={J2_}_J3={J3_}_rmax={r_max}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.2_HypGRU_70_id_hyp_rmax=1.0_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU, numsamples, h_wl2, Ee_02_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-42.793701171875-0.008200000040233135j), var E = 1.281499981880188
DMRG energy  is -43.586
Time taken=0.821 hrs


In [15]:
units = 70
seed=555
hGRU = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max = r_max, seed=seed)
h_wl2 = f'J2={J2_}_J3={J3_}_rmax={r_max}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.2_HypGRU_70_id_hyp_rmax=1.0_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU, numsamples, h_wl2, Ee_02_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 401 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-42.29679870605469+0.0015999999595806003j), var E = 0.8694000244140625
DMRG energy  is -43.586
Time taken=0.446 hrs


## J2=0.2, J3 = 0.5

In [15]:
J2_ = 0.2
J3_ = 0.5
J2 = +J2_ * np.ones(syssize)
J3 = +J3_ * np.ones(syssize)
Ee_02_05=-49.628675088072384

In [11]:
units = 70
r_max=0.95
seed=111
set_cpu_deterministic(seed)
hGRU = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max = r_max, seed=seed)
h_wl2 = f'J2={J2_}_J3={J3_}_rmax={r_max}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.5_HypGRU_70_id_hyp_rmax=0.95_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU, numsamples, h_wl2, Ee_02_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-48.989498138427734-0.0017000000225380063j), var E = 2.113300085067749
DMRG energy  is -49.6287
Time taken=0.619 hrs


In [24]:
units = 70
r_max=0.95
seed=222
hGRU = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max = 0.95, seed=seed)
h_wl2 = f'J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.5_HypGRU_70_id_hyp_rmax=0.95_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU, numsamples, h_wl2, Ee_02_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-47.81480026245117+0.001500000013038516j), var E = 3.3594000339508057
DMRG energy  is -49.6287
Time taken=0.945 hrs


In [25]:
units = 70
r_max=0.95
seed=333
hGRU = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max = 0.95, seed=seed)
h_wl2 = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.5_HypGRU_70_id_hyp_rmax=0.95_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU, numsamples, h_wl2, Ee_02_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-48.34560012817383-0.005100000184029341j), var E = 2.9217000007629395
DMRG energy  is -49.6287
Time taken=0.716 hrs


In [33]:
units = 70
r_max=0.95
seed=444
hGRU = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max = 0.95, seed=seed)
h_wl2 = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.5_HypGRU_70_id_hyp_rmax=0.95_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU, numsamples, h_wl2, Ee_02_05, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 391 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-48.911598205566406+0.00430000014603138j), var E = 0.7835999727249146
DMRG energy  is -49.6287
Time taken=0.508 hrs


In [34]:
units = 70
r_max=0.95
seed=555
hGRU = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max = 0.95, seed=seed)
h_wl2 = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.5_HypGRU_70_id_hyp_rmax=0.95_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU, numsamples, h_wl2, Ee_02_05, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 416 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-47.81890106201172-0.0005000000237487257j), var E = 0.9372000098228455
DMRG energy  is -49.6287
Time taken=0.527 hrs


## J2=0.5, J3=0.2

In [12]:
J2_ = 0.5
J3_= 0.2
J2 = +J2_ * np.ones(syssize)
J3 = +J3_ * np.ones(syssize)
Ee_05_02= -38.54733450537742

In [14]:
units = 70
r_max=0.7
seed=111
set_cpu_deterministic(seed)
hGRU = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max = r_max, seed=seed)
h_wl2 = f'J2={J2_}_J3={J3_}_rmax={r_max}/seed={seed}/N100_J1=1.0|J2={J2_}|J3={J3_}_HypGRU_70_id_hyp_rmax={r_max}_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU, numsamples, h_wl2, Ee_05_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 517 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-38.31420135498047+0.00039999998989515007j), var E = 0.26969999074935913
DMRG energy  is -38.5473
Time taken=0.563 hrs


In [27]:
units = 70
r_max=0.7
seed=333
hGRU = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max = r_max, seed=seed)
h_wl2 = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.5|J3=0.2_HypGRU_70_id_hyp_rmax=0.7_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU, numsamples, h_wl2, Ee_05_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-36.727699279785156-0.002300000051036477j), var E = 3.731100082397461
DMRG energy  is -38.5473
Time taken=0.564 hrs


In [23]:
units = 70
r_max=0.7
seed=444
hGRU = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max = r_max, seed=seed)
h_wl2 = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.5|J3=0.2_HypGRU_70_id_hyp_rmax=0.7_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU, numsamples, h_wl2, Ee_05_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 632 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-36.56930160522461-0.006000000052154064j), var E = 2.921600103378296
DMRG energy  is -38.5473
Time taken=0.519 hrs


In [29]:
units = 70
r_max=0.7
seed=555
hGRU = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max = r_max, seed=seed)
h_wl2 = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.5|J3=0.2_HypGRU_70_id_hyp_rmax=0.7_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU, numsamples, h_wl2, Ee_05_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.30459976196289+0.003100000089034438j), var E = 2.6647000312805176
DMRG energy  is -38.5473
Time taken=0.594 hrs
